# StockSense - Notebook 01: Data Audit, Quality & Cleaning
**Role**: Student 1 (Data Analyst) | Team CodeHawks  
**Objective**: Audit 5 raw NovaMart datasets, identify data quality traps, execute reproducible cleaning, and assemble the unified master table at date x store x product grain.


## Section 1: Environment Setup & Configurations
**Business Question**: How do we establish a reproducible, path-agnostic data pipeline with deterministic random seeds?
**What We Do**: Import `src/config.py` constants, initialize system paths, and set pandas formatting.
**What We Found**: Environment initialized with `RANDOM_SEED = 42`.


In [1]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
from src.config import RAW_DIR, PROCESSED_DIR, REPORTS_DIR
from src.data_cleaning import (
    clean_products, clean_stores, clean_transactions,
    clean_external_factors, clean_inventory, get_quality_log_dataframe
)
from src.build_master import build_master_table

pd.set_option('display.max_columns', 30)
print("Setup complete. Raw data path:", RAW_DIR)


Setup complete. Raw data path: C:\Users\Sujay\Documents\STOCKSENSE_CODEHAWKS\data\raw


## Section 2: Raw Data Audit (Part A)
**Business Question**: What intentional data traps (missing values, mixed date formats, duplicate keys, arithmetic errors) exist in raw files?
**What We Do**: Perform deep structural audit across `transactions.csv`, `products.csv`, `stores.csv`, `inventory.csv`, and `external_factors.csv`.
**What We Found**: Audited 174,613 raw transactions, 37 products, 6 stores, 22,503 inventory logs, and 732 weather records.


In [2]:
# Audit raw files
raw_tx = pd.read_csv(RAW_DIR / 'transactions.csv')
raw_prod = pd.read_csv(RAW_DIR / 'products.csv')
raw_stores = pd.read_csv(RAW_DIR / 'stores.csv')
raw_inv = pd.read_csv(RAW_DIR / 'inventory.csv')
raw_ext = pd.read_csv(RAW_DIR / 'external_factors.csv')

print("--- RAW DATA SHAPES ---")
print(f"Transactions: {raw_tx.shape} | Duplicates: {raw_tx.duplicated().sum()}")
print(f"Products:     {raw_prod.shape} | Duplicates: {raw_prod.duplicated().sum()}")
print(f"Stores:       {raw_stores.shape} | Duplicates: {raw_stores.duplicated().sum()}")
print(f"Inventory:    {raw_inv.shape} | Duplicates: {raw_inv.duplicated().sum()}")
print(f"External:     {raw_ext.shape} | Duplicates: {raw_ext.duplicated().sum()}")


--- RAW DATA SHAPES ---
Transactions: (174613, 11) | Duplicates: 1041
Products:     (37, 8) | Duplicates: 1
Stores:       (6, 6) | Duplicates: 0
Inventory:    (22503, 9) | Duplicates: 10
External:     (732, 8) | Duplicates: 6


## Section 3: Data Cleaning Execution (Part B)
**Business Question**: How do we clean, impute, and standardize raw files without introducing data leakage or dropping valid customer demand?
**What We Do**: Execute modular functions in `src/data_cleaning.py` to fix dates (`DD-MM-YYYY`), resolve inventory mismatches, segregate returns/outliers, and impute discounts/temperatures.
**What We Found**: Dropped 1,041 true duplicate transactions, segregated 45 returns + 6 outliers, reconciled 377 inventory log issues, and standardized product categories to 7 canonical classes.


In [3]:
# Execute cleaning module
clean_p = clean_products(raw_prod)
clean_s = clean_stores(raw_stores)
clean_t = clean_transactions(raw_tx, clean_p)
clean_e = clean_external_factors(raw_ext, clean_s)
clean_i = clean_inventory(raw_inv, clean_t)

log_df = get_quality_log_dataframe()
print(f"Data Quality Issues Logged: {len(log_df)} entries.")
log_df.head(10)


Data Quality Issues Logged: 21 entries.


,file,issue,count,action,justification
0,products.csv,Duplicate product rows,1,Dropped duplicate rows,Product metadata should be unique per product_id.
1,products.csv,Inconsistent category names & casing,14,Mapped to 7 standard Title Case categories usi...,Standardization is required for clean aggregat...
2,products.csv,Missing shelf_life_days,1,Imputed using median shelf life of sub_categor...,Sub-category peers share similar physical expi...
3,products.csv,Missing brand name,1,Filled missing brand with 'Unknown',Prevents missing values while preserving produ...
4,stores.csv,Store ID and text formatting,0,Validated store IDs and region mappings,Store metadata is clean and intact across all ...
5,transactions.csv,True duplicate transactions,1041,Dropped identical duplicate transaction rows,Duplicate rows distort demand volume and total...
6,transactions.csv,Mixed date formats (DD-MM-YYYY vs YYYY-MM-DD),520,Parsed DD-MM-YYYY explicitly with format %d-%m-%Y,Prevents month/day swap errors during time-ser...
7,transactions.csv,Negative quantity returns,45,Excluded from demand calculations and moved to...,"Customer returns represent reverse logistics, ..."
8,transactions.csv,Extreme quantity outliers (>100 units/basket),6,Excluded from demand calculations and moved to...,Bulk wholesale orders warp retail store demand...
9,transactions.csv,Missing discount_pct values,1735,"Imputed: 1720 via store-product-date mean, 15 ...",Accurate discount percentages are vital for pr...


## Section 4: Master Table Construction & Contract Assertions
**Business Question**: How do we build a clean master dataset at ONE ROW = ONE DATE x ONE STORE x ONE PRODUCT grain that passes all strict data contract checks?
**What We Do**: Run `src/build_master.py` to merge inventory grid, sales aggregates, product/store metadata, and external drivers into `master_table.csv`.
**What We Found**: Generated **22,523 rows**, 6 stores, 36 products, 344,525 total units sold, Rs. 22.34M total revenue, and **7.80% stock-out rate**.


In [4]:
# Build master table and verify contract assertions
master_df = build_master_table()
print("Master Table Head:")
master_df.head(5)


Reading raw datasets from: C:\Users\Sujay\Documents\STOCKSENSE_CODEHAWKS\data\raw


Step 1: Cleaning products metadata...
Step 2: Cleaning stores metadata...
Step 3: Cleaning transaction records...


Step 4: Cleaning external factor records...
Step 5: Cleaning inventory logs...


Step 6: Aggregating transaction records to Daily Store x Product grain...
Step 7: Merging daily transaction aggregates onto inventory grid...
Step 8: Merging product, store, and external factor metadata...
Step 9: Adding stockout flags and history metrics...
Step 10: Executing rigorous data contract assertion checks...
MASTER TABLE VERIFIED: 22523 rows, 6 stores, 36 products.
Total Units Sold: 344,525 | Total Revenue: Rs. 22,343,791.85
Stock-out Day Rate: 7.80%


Saved master table to: C:\Users\Sujay\Documents\STOCKSENSE_CODEHAWKS\data\processed\master_table.csv
Saved data quality log to: C:\Users\Sujay\Documents\STOCKSENSE_CODEHAWKS\reports\data_quality_log.csv
Master Table Head:


,date,store_id,product_id,opening_stock,received,sold,closing_stock,reorder_lvl,lead_days,inventory_fixed_flag,units_sold,revenue,avg_discount_pct,promotion_flag,txn_count,...,city,store_type,floor_area_sqft,avg_daily_customers,region,temp_c,rain_mm,holiday,festival,weekend,local_event,temp_imputed_flag,stockout_flag_day,history_days,is_sparse_history
0,2026-05-01,S01,P102,59,0,10,49,37,1,0,10,340.0,0.0,0,4,...,Coimbatore,Supermarket,8500,1250,West,32.5,0.0,1,0,0,0,0,0,1,0
1,2026-05-02,S01,P102,49,0,15,34,37,1,0,15,510.0,0.0,0,9,...,Coimbatore,Supermarket,8500,1250,West,33.4,11.6,0,0,1,0,0,0,2,0
2,2026-05-03,S01,P102,34,25,36,23,37,1,0,36,1224.0,0.0,0,16,...,Coimbatore,Supermarket,8500,1250,West,31.5,0.0,0,0,1,0,0,0,3,0
3,2026-05-04,S01,P102,23,0,18,5,37,1,0,18,612.0,0.0,0,7,...,Coimbatore,Supermarket,8500,1250,West,33.3,0.0,0,0,0,0,0,0,4,0
4,2026-05-05,S01,P102,5,36,26,15,37,1,0,26,884.0,0.0,0,13,...,Coimbatore,Supermarket,8500,1250,West,33.7,0.0,0,0,0,0,0,0,5,0
